# 📗 서빙 메트릭 모니터링: 서버가 기록한 숫자로 처리 상태와 운영 원가 읽기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">이 교안을 배우는 이유</strong></p>
<p style="margin:6px 0;">서버가 켜져 있고 질문 한 건에 답해도, 요청이 몰릴 때 얼마나 빨리 답하고 비용이 얼마나 드는지는 알 수 없습니다. 이 교안에서는 서버가 기록한 요청 수·응답 시간·대기 상태를 읽어 운영 상태와 원가를 확인합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/09_why_metrics.png" width="1050" alt="서버가 켜졌는지 확인한 다음에도 요청 수, 응답 시간, GPU 사용 요금을 관측해야 하는 이유를 보여 줍니다.">

#### **메트릭(metric)**

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버의 처리 상태를 숫자로 기록한 값</strong>입니다. 지금까지 처리한 요청 수, 응답에 걸린 시간, 현재 기다리는 요청 수가 메트릭입니다.</p>
<p style="margin:6px 0;">예를 들어 서버의 요청 수가 100건에서 164건으로 늘었다면, 그사이에 64건을 처리한 것입니다. 이런 숫자로 서버가 얼마나 많은 요청을 받고, 얼마나 빨리 답하는지 확인합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/10_metrics_concept.png" width="1050" alt="메트릭의 요청 수 예시: 요청을 보내기 전 100건, 보낸 뒤 164건이 기록되어 이번에 처리한 요청이 64건임을 보여 줍니다.">

#### `/metrics` 조회 주소

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>는 서버가 기록한 메트릭 숫자를 읽는 주소입니다. 상담 주소에 질문을 보내면 답변이 오고, 이 주소를 읽으면 요청 수와 응답 시간 같은 운영 숫자가 옵니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">

<span class="day79-goal" style="font-size:1.13em;color:#1f3a5f;">**학습 목표**</span>

1. 서빙 API에 요청 수, 걸린 시간, 출력 토큰 메트릭을 붙여 배포 파이프라인으로 배포합니다.
2. 게이지, 카운터, 히스토그램을 구분하고, 두 시점의 차이로 부하 구간의 값을 계산합니다.
3. vLLM의 포화 신호(대기 요청, KV 캐시 사용률)를 읽고, 히스토그램 경계로 경계 이하 비율(15초 이하, TTFT(첫 토큰 지연) 1초 이하)을 정확히 계산합니다.
4. 구간의 처리량과 100만 출력 토큰당 운영 원가를 계산하고, 서빙 API 층과 vLLM 층의 숫자가 다른 이유를 설명합니다.

</div>

## 1. 오늘 시작 상태

### 1-1. 저장한 템플릿으로 GPU Pod 만들기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><a style="color:#245fa0;text-decoration:underline;" href="https://console.runpod.io/pods">RunPod 콘솔</a>에서 <strong style="font-size:1.13em;color:#1f3a5f;">어제 설정한 템플릿으로 GPU Pod 하나를 만듭니다.</strong> 템플릿 설정은 그대로 씁니다.</p>
<p style="margin:6px 0;">만든 Pod의 <strong style="font-size:1.13em;color:#1f3a5f;">ID</strong>와 <strong style="font-size:1.13em;color:#1f3a5f;">시간당 요금(Total / hour)</strong>을 확인하고, 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">실습자료/.env</code>에서 다음 두 줄을 바꿉니다.</p>
</div>

```dotenv
GPU_POD_ID=<새 Pod의 ID>
GPU_PRICE_PER_HOUR=<Total / hour의 달러 금액, 예: 0.58>
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">GPU_POD_ID</code>:</strong> 어제 종료한 Pod의 ID를 새 ID로 바꿉니다. 노트북이 이 값으로 새 GPU 서버 주소를 만듭니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">GPU_PRICE_PER_HOUR</code>:</strong> 운영 원가 계산에 쓸 숫자만 적습니다. 예를 들어 시간당 <span>&#36;</span>0.58이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">0.58</code>입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>, <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">ADMIN_API_KEY</code>:</strong> 기존 값을 그대로 둡니다.</li>
</ul>
</div>

### 1-2. 노트북 준비

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">배포 단원과 같은 준비 코드(경로, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code> 읽기)에 JSONL 읽기 함수와 노무 질문을 더해 한 셀에 모았습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 이 셀에서 읽으므로, 위에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_POD_ID</code>를 저장한 뒤 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 읽고 GPU·VM 주소, 노무 질문, 부하와 메트릭 조회에 필요한 준비값을 만듭니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 설정값(배포 단원과 같습니다) ----
import json
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

day_dir = Path("..").resolve()       # data, images, output 이 있는 일차 폴더(day79_서빙운영_관측_SLO)
data_dir = day_dir / "data"
output_dir = day_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 키와 주소는 실습자료/.env 에 모아 둡니다(화면에 출력하지 않습니다)
env_path = day_dir.parent / ".env"
load_dotenv(env_path, override=True)

SERVED_NAME = "qwen3.5-4b"            # vLLM의 --served-model-name

# 테이블의 긴 글(질문, 답)이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# ---- JSONL 읽기 ----
def load_jsonl(path):
    """path가 가리키는 UTF-8 JSONL 파일을 읽어 딕셔너리 목록으로 돌려줍니다.

    파일의 각 줄을 JSON 딕셔너리 하나로 바꾸고, 원래 줄 순서를 유지해 리스트에 담습니다.
    질문 파일은 보통 한 딕셔너리에 question과 기대하는 메타데이터를 담습니다.
    """
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# ---- 노무 질문(서빙 벤치마킹 단원부터 쓴 같은 파일) ----
labor_questions = load_jsonl(data_dir / "본실습" / "labor_questions.jsonl")

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 교안이 쓰는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code> 값이 있는지 이름만 확인합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_PRICE_PER_HOUR</code>는 1-1에서 적었고, 나머지는 배포 단원에서 적었습니다.</p>
</div>

In [ ]:
# .env 에 필요한 값이 있는지 이름만 확인합니다(값은 출력하지 않습니다)
env_check = {}
for name in ["VLLM_API_KEY", "ADMIN_API_KEY", "GPU_POD_ID", "GPU_PRICE_PER_HOUR", "VM_HOST", "GITHUB_OWNER"]:
    env_check[name] = name in os.environ
display(env_check)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">/metrics</code>는 서버가 기록한 숫자를 읽는 조회 주소입니다.</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/labor/chat</code>에 질문을 보내면 답변이 오고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>를 읽으면 요청 수와 응답 시간 같은 숫자가 옵니다. 아래 셀은 서빙 API와 vLLM의 조회 주소를 만듭니다.</p>
<p style="margin:6px 0;">주소와 헤더는 배포 단원과 같은 이름으로 만듭니다.</p>
</div>

In [ ]:
# GPU 파드의 vLLM 주소(RunPod proxy)와 vLLM 키 헤더(배포 단원 교안 02 와 같은 이름)
VLLM_URL = "https://" + os.environ["GPU_POD_ID"] + "-8000.proxy.runpod.net/v1"
vllm_headers = {"Authorization": "Bearer " + os.environ["VLLM_API_KEY"]}

# VM 의 서빙 API 주소와 관리자 키 헤더
VM_URL = "http://" + os.environ["VM_HOST"] + ":8080"
ADMIN_HEADERS = {"X-Admin-Key": os.environ["ADMIN_API_KEY"]}

# 두 층의 /metrics 주소. 서빙 API 는 관리자 키가 있어야 열리고, vLLM 은 키 없이 열린다(/v1 로 시작하지 않는 경로)
SERVING_METRICS_URL = VM_URL + "/metrics"
VLLM_METRICS_URL = "https://" + os.environ["GPU_POD_ID"] + "-8000.proxy.runpod.net/metrics"

# 노트북이 VM 의 서빙 API 응답을 기다리는 최대 시간(초). VM 의 vLLM 시간 제한 90초보다 길게 둡니다
REQUEST_TIMEOUT = 120

### 1-3. 서빙 API를 새 GPU Pod에 연결

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM에는 애플리케이션 코드를 두지 않습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">compose.yaml</code>은 컨테이너 실행 설정만 담고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>는 GPU 주소와 키를 담습니다.</p>
<p style="margin:6px 0;">아래 셀은 새 GPU Pod 주소를 넣은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>를 만듭니다. 다음 셀은 VM의 기존 폴더를 지우고 두 파일만 복사합니다. 서버 코드를 push하면 GitHub Actions가 GitHub Container Registry(GHCR)에서 이미지를 받아 VM에서 실행합니다.</p>
</div>

In [ ]:
# VM 서빙 API가 읽을 환경변수. 키와 이미지 이름은 그대로 두고 GPU 서버 주소만 바꿉니다.
vm_env_lines = [
    "SERVING_IMAGE=ghcr.io/" + os.environ["GITHUB_OWNER"].lower() + "/serving-deploy",
    "VLLM_BASE_URL=" + VLLM_URL,
    "VLLM_API_KEY=" + os.environ["VLLM_API_KEY"],
    "ADMIN_API_KEY=" + os.environ["ADMIN_API_KEY"],
    "VLLM_TIMEOUT_SECONDS=90",
]
vm_serving_env = output_dir / "vm_serving.env"
vm_serving_env.write_text("\n".join(vm_env_lines) + "\n", encoding="utf-8")

# 키 값은 출력하지 않고 저장한 이름만 확인합니다.
written_names = [line.split("=")[0] for line in vm_env_lines]
display(written_names)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">내 PC 터미널에서 기존 폴더를 지우고 배포 파일 두 개만 보냅니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
ssh serving-vm 'rm -rf ~/serving-deploy && mkdir -p ~/serving-deploy/deploy'
scp day79_서빙운영_관측_SLO/교안_01_서빙_메트릭_모니터링/serving-deploy/deploy/compose.yaml serving-vm:~/serving-deploy/deploy/compose.yaml
scp day79_서빙운영_관측_SLO/output/vm_serving.env serving-vm:~/serving-deploy/deploy/.env
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">코드를 push하면 GitHub Actions가 GHCR 이미지를 받아 VM의 Compose로 실행합니다.</p>
</div>

## 2. 서빙 API에 메트릭 붙이기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">파드가 모델을 받는 동안 서버 코드를 고쳐 배포합니다. 바뀐 파일은 이 교안 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy/</code>에 있고, 다른 파일은 배포 단원과 같습니다.</p>
</div>

### 2-1. 바뀐 파일 읽기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app/core/metrics.py</code>가 새 파일입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prometheus_client</code>는 요청 수와 응답 시간을 기록하고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>에서 읽을 수 있게 해 주는 파이썬 라이브러리입니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Counter</code>:</strong> 요청이 끝날 때마다 개수를 더합니다. 요청 수처럼 계속 쌓이는 값을 기록합니다.</p>
</li>
<li style="margin:3px 0;"><p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Histogram</code>:</strong> 응답 시간이 1초 이하인지, 5초 이하인지처럼 정해 둔 경계별로 요청 수를 셉니다.</p>
</li>
<li style="margin:3px 0;"><p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">라벨(label):</strong> 같은 숫자를 주소나 상태 코드별로 나눠 세기 위한 꼬리표입니다. 예를 들어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">status=&quot;200&quot;</code>은 정상 응답만 고르는 표시입니다.</p>
</li>
<li style="margin:3px 0;"><p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Prometheus:</strong> 여러 서버의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>를 주기적으로 읽어 모아 두는 오픈소스 모니터링 도구입니다. 이 교안은 Prometheus를 설치하지 않고, 그 도구가 읽는 텍스트 형식만 노트북에서 직접 읽습니다.</p>
</li>
<li style="margin:3px 0;"><p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">Prometheus 텍스트 형식:</strong> 숫자마다 이름과 라벨을 붙여 한 줄씩 적는 형식입니다. 예를 들어 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">requests_total{status=&quot;200&quot;} 3</code>은 정상 응답이 3건이라는 뜻입니다. 실제 메트릭의 이름은 실행 결과에서 확인합니다.</p>
</li>
</ul>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving_app/core/metrics.py</strong></div>

```python
"""서빙 API 의 메트릭(관측 단원 교안 01). 상담 요청 수, 걸린 시간, 출력 토큰 수를 세어 /metrics 로 내보낸다.

prometheus_client 가 숫자를 메모리에 세어 두고, /metrics 를 부르면 그 숫자를 Prometheus 텍스트 형식으로 돌려준다.
vLLM 의 /metrics 와 같은 형식이라 같은 방법으로 읽는다.
"""
import time

from fastapi import APIRouter, Depends, Response
from prometheus_client import CONTENT_TYPE_LATEST, Counter, Histogram, disable_created_metrics, generate_latest

from ..dependencies.security import require_admin

# 걸린 시간을 측정하는 상담 주소. 아무 주소나 세면 없는 주소로 보낸 요청마다 줄이 늘어나므로, 정해 둔 주소만 센다
# 스트리밍 주소는 첫 조각이 나갈 때 응답이 시작되어 전체 시간을 측정할 수 없으므로 넣지 않는다
CHAT_PATHS = ["/v1/labor/chat", "/v1/lease/chat"]

# Counter, Histogram, Summary 는 _created 보조 메트릭도 내보낼 수 있습니다.
# _created 의 숫자는 메트릭을 만든 Unix 시각이며 요청 수가 아닙니다. 이 실습에서는 쓰지 않아 내보내지 않습니다.
disable_created_metrics()

# 카운터: 늘어나기만 하는 수. /metrics 에는 이름 뒤에 _total 이 붙어 나간다
REQUESTS = Counter("serving_requests", "상담 주소가 돌려준 응답 수", ["path", "status"])

# Histogram 은 요청 시간을 정한 경계별 누적 건수로 기록합니다.
# buckets 는 초 단위 상한 목록이며, 15를 넣으면 15초 이하 요청 수를 셀 수 있습니다.
REQUEST_SECONDS = Histogram(
    "serving_request_seconds",
    "상담 요청 하나를 받아 응답하기까지 걸린 시간(초)",
    ["path"],
    buckets=[0.5, 1, 2.5, 5, 10, 15, 20, 30, 60, 90],
)

# 창구별 출력 토큰 수(GPU 가 만든 토큰). 상담 엔드포인트가 답을 받은 뒤 더한다
OUTPUT_TOKENS = Counter("serving_output_tokens", "창구별 출력 토큰 수", ["desk"])

metrics_router = APIRouter(tags=["관측"])


# GET /metrics 주소를 등록하고, Depends(require_admin) 이 관리자 키를 확인합니다.
@metrics_router.get("/metrics", dependencies=[Depends(require_admin)])
def metrics():
    """관리자 키를 확인한 뒤, 현재 등록된 메트릭을 Prometheus 텍스트로 돌려줍니다."""
    # generate_latest() 는 등록된 메트릭을 모아 Prometheus 텍스트 본문으로 만듭니다.
    metrics_text = generate_latest()
    # Response 는 HTTP 응답을 만듭니다. content 는 본문, media_type 은 본문 형식을 알리는 Content-Type 헤더입니다.
    return Response(content=metrics_text, media_type=CONTENT_TYPE_LATEST)


async def record_chat_request(request, call_next):
    """모든 HTTP 요청을 다음 처리 단계로 넘기고, 상담 주소면 처리 시간과 응답 상태를 기록합니다."""
    # request.url.path 는 호스트와 쿼리를 뺀 경로입니다. 측정 대상이 아니면 다음 처리 단계로 넘깁니다.
    path = request.url.path
    if path not in CHAT_PATHS:
        # call_next(request) 가 요청을 나머지 앱으로 보내고, 그 앱의 응답을 돌려줍니다.
        return await call_next(request)

    # time.perf_counter() 값을 응답 전후로 빼 요청 처리에 걸린 초를 구합니다.
    started = time.perf_counter()
    # call_next(request) 가 요청을 인증·제한·상담 처리기로 보내고, 만들어진 응답을 돌려줍니다.
    response = await call_next(request)
    elapsed_seconds = time.perf_counter() - started
    # labels(path=...) 는 이 주소의 히스토그램을 고르고, observe(...) 는 걸린 초를 기록합니다.
    REQUEST_SECONDS.labels(path=path).observe(elapsed_seconds)
    # labels(path, status) 는 주소와 상태 코드별 카운터를 고르고, inc() 는 완료 요청 하나를 더합니다.
    REQUESTS.labels(path=path, status=str(response.status_code)).inc()
    return response
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">미들웨어(middleware)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><p style="margin:6px 0;">앱이 받는 모든 HTTP 요청이 엔드포인트에 닿기 전과 응답이 나간 뒤에 거치는 함수입니다. FastAPI가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">call_next</code>에 다음 처리 함수를 넣어 줍니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">call_next(request)</code>는 요청을 나머지 앱에 넘기고, 그 앱이 돌려준 응답을 반환합니다.</p>
</li>
<li style="margin:3px 0;"><p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">메트릭을 미들웨어에서 세는 이유:</strong> 정상 응답(200)과, 앱에 등록한 오류 처리기(FastAPI 서빙 API 단원의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">register_error_handlers</code>)가 만든 응답(401 키 없음, 422 양식 오류, 429 한도 초과, 502, 504)을 한 곳에서 함께 셉니다.</p>
</li>
<li style="margin:3px 0;"><p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">세지 않는 것:</strong> 오류 처리기가 받지 않는 예외(코드 버그로 생긴 500)는 응답이 되기 전에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">call_next</code>를 뚫고 올라가므로 이 미들웨어가 세지 못합니다.</p>
</li>
<li style="margin:3px 0;"><p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">세 메트릭:</strong> 요청 수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_requests</code>(라벨 path, status), 걸린 시간 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_request_seconds</code>(라벨 path), 출력 토큰 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_output_tokens</code>(라벨 desk)입니다. 라벨은 같은 메트릭을 나눠 세는 기준으로, path는 주소별, status는 상태 코드별로 따로 셉니다(3-2에서 원문으로 봅니다).</p>
</li>
<li style="margin:3px 0;"><p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">buckets</code>:</strong> 요청 시간을 비교할 초 단위 기준값 목록입니다. 예를 들어 15를 넣으면 15초 이하로 끝난 요청 수를 셀 수 있습니다. 경계별 숫자가 어떻게 쌓이는지는 아래 예에서 봅니다.</p>
</li>
</ul>
<p style="margin:6px 0;">앱에는 미들웨어와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code> 주소를 붙이는 두 줄이 더해졌습니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving_app/managed_main.py(바뀐 줄)</strong></div>

```python
# managed_main.py
from .core.metrics import metrics_router, record_chat_request

# 관측 단원: 모든 HTTP 요청이 record_chat_request 를 거치게 하고(미들웨어), /metrics 주소를 붙인다
app.middleware("http")(record_chat_request)
app.include_router(metrics_router)
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">출력 토큰 수는 vLLM의 답이 와야 알 수 있으므로 미들웨어가 아니라 상담 엔드포인트 안에서 더합니다. 스트리밍 주소도 마지막 usage 이벤트에서 같은 카운터에 더합니다.</p>
</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">desk</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/{desk}/chat</code> 주소에서 정해지는 창구 이름이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">completion</code>은 vLLM이 돌려준 답입니다. 아래는 답을 만든 뒤 출력 토큰 수를 기록하는 라우터 함수 전체입니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · serving_app/routers/metered_router.py (상담 함수 전체)</strong></div>

```python
# routers/metered_router.py
# 경로 매개변수 desk 에 Literal 을 붙여 노무, 전월세 창구를 엔드포인트 하나로 처리한다
@metered_router.post("/{desk}/chat", response_model=ChatResponse)
async def metered_chat(
    desk: Literal["labor", "lease"],
    request: ChatRequest,
    client: dict = Depends(check_rate_limit),
    connection=Depends(get_db),  # lifespan 이 연 DB 연결
):
    """상담 답을 돌려주고, 키별 사용량과 창구별 출력 토큰 수를 기록합니다."""
    # 1) desk 에 맞는 시스템 메시지로 vLLM 에 묻는다(desk 가 labor, lease 가 아니면 422)
    completion = await ask_vllm(SYSTEM_PROMPTS[desk], request.question, request.think, max_tokens_think=THINK_LIMITS[desk])

    # 2) 그 요청의 토큰 사용량을 기록한다. client 는 check_rate_limit 을 통과한 키다
    usage = completion.usage
    await record_usage(
        connection, client["key_id"], f"/v1/{desk}/chat",
        usage.prompt_tokens, usage.prompt_tokens_details.cached_tokens, usage.completion_tokens,
    )

    # labels(desk=desk) 는 이 창구의 카운터를 고릅니다. inc(...) 는 이 답의 출력 토큰 수를 더합니다.
    OUTPUT_TOKENS.labels(desk=desk).inc(completion.usage.completion_tokens)
    return to_chat_response(completion)
```

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · pyproject.toml(더한 줄)</strong></div>

```toml
# pyproject.toml
    "prometheus-client>=0.26.0",   # /metrics 로 내보낼 숫자 세기(관측 단원 교안 01)
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv.lock</code>도 이 라이브러리를 담아 바뀌었습니다. 이미지 테스트 파일 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tests/test_api.py</code>에는 메트릭 테스트 두 개를 더했습니다. ci의 image 작업은 컨테이너를 띄운 뒤 이 파일만 실행하므로, 새 파일을 만들지 않고 이 파일 끝에 더해야 CI가 함께 확인합니다. 키 없는 상담 요청 한 건(401)을 보내고, 미들웨어가 그 응답을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>에 세었는지 봅니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · tests/test_api.py(더한 줄)</strong></div>

```python
# test_api.py
ADMIN_API_KEY = os.environ["ADMIN_API_KEY"]                       # 관측 단원: /metrics 를 여는 관리자 키(CI 의 image 작업이 준다)


# 관측 단원 교안 01: 메트릭 주소 테스트 두 개
def test_metrics_needs_admin_key():
    # 관리자 키 없이 /metrics 를 부르면 403
    response = httpx.get(f"{SERVING_URL}/metrics", timeout=10)
    assert response.status_code == 403


def test_chat_is_counted():
    # 키 없는 상담 요청(401)도 미들웨어가 세므로, /metrics 에 그 상태 코드로 센 줄과 15초 경계 칸이 있어야 한다
    httpx.post(f"{SERVING_URL}/v1/labor/chat", json=QUESTION, timeout=10)
    response = httpx.get(f"{SERVING_URL}/metrics", headers={"X-Admin-Key": ADMIN_API_KEY}, timeout=10)
    assert response.status_code == 200
    assert 'serving_requests_total{path="/v1/labor/chat",status="401"}' in response.text
    assert 'serving_request_seconds_bucket{le="15.0",path="/v1/labor/chat"}' in response.text
```

### 2-2. 저장소에 복사해 올리기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">실습자료 폴더에서 전체 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-deploy</code> 저장소를 내 PC의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/serving-deploy/</code>에 복사합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pushd</code>로 저장소에 들어가 커밋과 push를 한 뒤 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">popd</code>로 실습자료 폴더에 돌아옵니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
cp -R day79_서빙운영_관측_SLO/교안_01_서빙_메트릭_모니터링/serving-deploy/. ~/serving-deploy/
pushd ~/serving-deploy

git add .
git commit -m "서빙 API에 메트릭 붙이기"
git push
popd
```

### 2-3. 배포 확인

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">[GitHub Actions](https://github.com/&lt;GitHub 아이디&gt;/serving-deploy/actions)에서 이번 커밋의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ci</code>와 이어지는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd</code>가 모두 성공했는지 확인합니다.</p>
</div>

## 3. 메트릭 한 줄과 값의 세 종류

### 3-1. 서빙 API의 /metrics 받아 보기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서버는 요청을 처리할 때 요청 수와 응답 시간을 메모리에 기록합니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>에 GET 요청을 보내면, 그 숫자들을 다음과 같은 <strong style="font-size:1.13em;color:#1f3a5f;">문자열</strong>로 돌려줍니다.</p>
</div>

```text
serving_requests_total{path="/v1/labor/chat",status="200"} 3.0
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">이 한 줄은 노무 상담의 정상 응답이 지금까지 3건이라는 뜻입니다. 아래 함수는 이 문자열을 계산하기 쉬운 표로 바꿉니다.</p>
<ol style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">httpx.get</code>: 서버에서 문자열을 받습니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">text_string_to_metric_families</code>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prometheus_client</code>가 제공하는 함수로, 문자열에서 메트릭 이름·라벨·숫자를 나눠 읽습니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pd.DataFrame</code>: 읽은 숫자 줄마다 한 행인 표를 만듭니다. 이렇게 한 시점에 읽어 둔 표를 <strong style="font-size:1.13em;color:#1f3a5f;">스냅샷(snapshot)</strong>이라고 합니다.</li>
</ol>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">read_metrics</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">url</code>이 필요한 이유는 서빙 API와 vLLM이 서로 다른 주소에서 메트릭을 내보내기 때문입니다. 서빙 API 주소에는 관리자 키 헤더를 전달하고, 관리자 키가 필요 없는 vLLM 주소에는 헤더를 생략합니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">metric_sum</code>은 이 표에서 메트릭 이름과 라벨 조건이 맞는 행의 값을 더합니다. 실제 원문과 표를 차례로 확인합니다.</p>
</div>

<div class="day79-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# prometheus_client가 제공하는 해석 함수입니다. 숫자가 적힌 문자열을 이름·라벨·값으로 나눕니다.
from prometheus_client.parser import text_string_to_metric_families


def read_metrics(url, headers=None):
    """url의 메트릭 조회 주소에 요청해 서버가 기록한 숫자를 표로 돌려줍니다.

    headers에는 관리자 키처럼 조회 권한에 필요한 HTTP 헤더를 넣습니다. 서버 응답은 JSON이 아니라
    Prometheus 형식의 텍스트이며, text_string_to_metric_families가 이를 메트릭 묶음(family)과
    숫자가 적힌 줄(sample)로 나눕니다. 반환 표는 숫자 줄마다 한 행이며 name(이름),
    labels(구분 라벨), value(숫자) 열을 가집니다. HTTP 오류가 오면 예외를 내고 멈춥니다.
    """
    # 서버의 조회 주소에 GET 요청을 보냅니다. 응답 본문은 JSON이 아니라 숫자가 적힌 문자열입니다.
    response = httpx.get(url, headers=headers, timeout=30)
    response.raise_for_status()  # 서버가 정상 응답하지 않았으면 여기서 오류를 알립니다.

    # 해석 함수가 같은 메트릭에 속한 줄을 family로 묶고, 각 숫자 줄을 sample로 제공합니다.
    rows = []
    for family in text_string_to_metric_families(response.text):
        for sample in family.samples:
            # 예: name=요청 수 메트릭 이름, labels={"status": "200"}, value=3.0
            rows.append({"name": sample.name, "labels": sample.labels, "value": sample.value})
    return pd.DataFrame(rows)  # 숫자 줄 하나를 표의 한 행으로 만듭니다.


def metric_sum(snapshot, name, **labels):
    """스냅샷 표에서 메트릭 이름과 라벨 조건이 맞는 숫자를 모두 더해 돌려줍니다.

    name에는 serving_requests_total처럼 메트릭 이름을 넣습니다. status="200" 또는
    path="/v1/labor/chat"처럼 필요한 라벨을 더 줄 수 있습니다. 라벨을 여러 개 주면
    모두 같은 행만 더하며, 조건에 맞는 행이 없으면 0을 돌려줍니다.
    """
    # **labels에는 status="200", path="/v1/labor/chat"처럼 넘긴 조건이 딕셔너리로 모입니다.
    total = 0.0
    for row in snapshot.itertuples():
        if row.name != name:  # 다른 메트릭은 건너뜁니다.
            continue
        # 요청한 라벨 조건이 모두 맞는지 확인합니다. 조건을 주지 않았다면 같은 이름의 행을 모두 더합니다.
        matched = True
        for key, value in labels.items():
            if row.labels.get(key) != value:
                matched = False
        if matched:
            total += row.value
    return total  # 맞는 행이 없으면 처음 값인 0입니다.

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 키 없는 상담은 거절되는지, 서빙 API의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>는 관리자 키 없이와 키를 넣었을 때 각각 어떻게 응답하는지 확인합니다.</p>
</div>

In [ ]:
# 서비스 키 없이 노무 상담 주소로 한 건 보냅니다(서빙 API 가 키 확인에서 401 로 답합니다)
no_key_chat = httpx.post(VM_URL + "/v1/labor/chat", json={"question": labor_questions[0]["question"]}, timeout=REQUEST_TIMEOUT)

# 관리자 키 없이, 관리자 키를 담아 /metrics 를 부릅니다
no_key_metrics = httpx.get(SERVING_METRICS_URL, timeout=30)
admin_metrics = httpx.get(SERVING_METRICS_URL, headers=ADMIN_HEADERS, timeout=30)
print("상담(키 없이):", no_key_chat.status_code, "| /metrics(키 없이):", no_key_metrics.status_code, "| /metrics(관리자 키):", admin_metrics.status_code)

In [ ]:
# 원문에서 서빙 API 가 센 줄만 봅니다(python_, process_ 로 시작하는 기본 줄은 뺍니다)
for line in admin_metrics.text.splitlines():
    if "serving_" in line:
        print(line)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">카운터와 히스토그램의 누적값은 서버가 실행되는 동안 메모리에 쌓입니다. 컨테이너를 다시 띄우면 0부터 시작합니다. 게이지는 누적하지 않고 읽은 순간의 현재 상태를 보여 줍니다. 라벨별 값 줄은 해당 라벨 조합으로 처음 요청을 처리한 뒤에 생깁니다.</p>
</div>

### 3-2. 이름, 라벨, 값

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>의 <strong style="font-size:1.13em;color:#1f3a5f;">값 줄 하나가 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">read_metrics</code> 표의 한 행</strong>이 됩니다.</p>
</div>

```text
# HELP serving_requests_total 상담 주소가 돌려준 응답 수
# TYPE serving_requests_total counter
serving_requests_total{path="/v1/labor/chat",status="401"} 1.0
```

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">표의 열</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">이 예시의 값</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">뜻</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">name</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_requests_total</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">응답 수를 세는 메트릭 이름</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">labels</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">path=&quot;/v1/labor/chat&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">status=&quot;401&quot;</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">노무 상담 주소에서 401 응답을 받은 요청을 구분</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">value</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">1.0</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">컨테이너가 시작된 뒤 이 주소·상태 조합에서 응답한 누적 횟수</td>
</tr>
</tbody>
</table>
<p style="margin:6px 0;">상태가 200인 응답은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">status=&quot;200&quot;</code>인 별도 줄에 셉니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># HELP</code>는 메트릭 이름의 뜻, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TYPE</code>은 메트릭 종류를 알려 줍니다. 둘 다 숫자 값 줄이 아니므로 표에 들어가지 않습니다.</p>
<p style="margin:6px 0;">아래에서 같은 주소의 값을 실제 표로 읽습니다.</p>
</div>

In [ ]:
# 같은 원문을 테이블로 읽고, 서빙 API 가 센 행만 봅니다
api_snapshot = read_metrics(SERVING_METRICS_URL, ADMIN_HEADERS)
display(api_snapshot[api_snapshot["name"].str.startswith("serving_")])

### 3-3. 카운터, 게이지, 히스토그램

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">메트릭의 값은 세 종류이고, 종류마다 읽는 법이 다릅니다(<a style="color:#245fa0;text-decoration:underline;" href="https://prometheus.io/docs/concepts/metric_types/">Prometheus 메트릭 종류</a>).</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/12_metric_types.png" width="1050" alt="메트릭 세 종류: 늘어나기만 하는 카운터, 오르내리는 게이지, 경계별 칸에 세는 히스토그램">

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">종류</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">뜻</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">/metrics의 이름</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">이 교안의 예</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">읽는 법</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><strong style="font-size:1.13em;color:#1f3a5f;">카운터(counter)</strong></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">늘어나기만 하는 누적 수. 서버가 다시 뜨면 0부터</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">끝에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_total</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_requests_total</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm:generation_tokens_total</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">두 시점의 차이</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><strong style="font-size:1.13em;color:#1f3a5f;">게이지(gauge)</strong></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">오르내리는 지금 값</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">이름 그대로</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm:num_requests_waiting</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm:kv_cache_usage_perc</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">읽은 순간의 값 그대로</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><strong style="font-size:1.13em;color:#1f3a5f;">히스토그램(histogram)</strong></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">관측값(걸린 시간)을 경계별 누적 수와 합으로 기록</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_bucket{le}</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_sum</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_count</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_request_seconds</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm:time_to_first_token_seconds</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">두 시점의 차이로 경계 이하 건수, 평균</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">buckets</code>:</strong> 초 단위 상한값 목록입니다. 요청이 15초 걸렸다면 15초와 그보다 큰 경계의 누적 수에 들어갑니다. 이전 경계와 현재 경계 사이에 든 요청만 따로 세는 방식이 아닙니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">le</code>(less than or equal):</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_bucket{le=&quot;15.0&quot;}</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">le</code>는 “이 값 이하”라는 뜻입니다. 이 줄에는 15초 이하 요청이 모두 들어가며, 더 큰 경계일수록 누적 수는 같거나 커집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">le=&quot;+Inf&quot;</code>:</strong> 모든 요청이 들어가는 마지막 경계라 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_count</code>와 같습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_sum</code>은 걸린 시간의 합이라 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_sum / _count</code>가 평균입니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">예:</strong> 노무 상담 네 건이 8초, 12초, 14초, 25초 걸렸다면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_request_seconds</code>는 아래처럼 남습니다. 12초와 14초는 &quot;10초 초과 15초 이하&quot; 칸에 들지만, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">le=&quot;15.0&quot;</code>은 8초까지 포함해 3입니다. 0.5~5초 경계(모두 0)와 60, 90초 경계(모두 4)는 줄였습니다.</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">줄</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">값</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_bucket{le=&quot;10.0&quot;}</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_bucket{le=&quot;15.0&quot;}</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_bucket{le=&quot;20.0&quot;}</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_bucket{le=&quot;30.0&quot;}</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_bucket{le=&quot;+Inf&quot;}</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_count</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">4</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_sum</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">59.0</td>
</tr>
</tbody>
</table>
</div>

### 3-4. vLLM의 /metrics

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM도 같은 형식의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>를 엽니다. 배포 단원에서 쓴 준비 상태 확인 셀로 GPU Pod가 요청을 받을 때까지 기다립니다.</p>
</div>

In [ ]:
import time

# 30초마다 모델 목록을 요청하고, 200 이면 멈춥니다(최대 40번, 약 20분)
for attempt in range(40):
    models_response = httpx.get(VLLM_URL + "/models", headers=vllm_headers, timeout=30)
    print(attempt + 1, "번째 요청:", models_response.status_code)
    if models_response.status_code == 200:
        break
    time.sleep(30)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM의 OpenAI 호환 서버는 자체 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code> 주소를 제공해 vLLM 메트릭을 내보냅니다. 이 엔드포인트는 이 교안에서 따로 구현하지 않아도 됩니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code> 요청에 쓰며, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>에는 적용되지 않아 헤더 없이 읽습니다. 준비 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_METRICS_URL</code>(proxy 주소 뒤에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1</code> 없이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>)로 접속해 이 교안이 읽는 항목만 골라 봅니다.</p>
</div>

In [ ]:
# vLLM 의 /metrics 를 키 없이 읽습니다. read_metrics 는 200 이 아니면 오류로 멈춥니다
vllm_snapshot = read_metrics(VLLM_METRICS_URL)

# 이 교안이 읽는 게이지와 카운터
watch_names = [
    "vllm:num_requests_running",
    "vllm:num_requests_waiting",
    "vllm:kv_cache_usage_perc",
    "vllm:generation_tokens_total",
    "vllm:prefix_cache_queries_total",
    "vllm:prefix_cache_hits_total",
    "vllm:request_success_total",
]
display(vllm_snapshot[vllm_snapshot["name"].isin(watch_names)])

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>는 키 없이 열립니다. 실제 서비스에서는 모델 서버를 내부 네트워크에 두고 메트릭을 수집합니다.</p>
</div>

## 4. 부하 보내고 구간 계산

### 4-1. 사용자 키 발급과 데움 요청

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">배포 단원에서 발급한 키는 원문이 그 노트북 변수에만 있었으므로 다시 쓸 수 없습니다. 같은 관리자 주소로 노무 부하에 쓸 사용자 세 명의 pro 키를 발급합니다. 발급 응답은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">issued_keys</code>에만 담고 키 원문은 화면에 출력하지 않습니다. 한 키는 60초에 30번까지 요청할 수 있습니다(FastAPI 서빙 API 단원 교안 02).</p>
</div>

In [ ]:
# 노무 부하에 쓸 사용자 세 명과 요금제
labor_users = ["labor-app", "hr-team", "mobile-app"]
labor_plans = {user: "pro" for user in labor_users}

# 관리자 주소로 세 사용자 키를 발급합니다. api_key 원문은 issued_keys 안에만 둡니다.
issued_keys = {}
labor_key_rows = []
for user, plan in labor_plans.items():
    response = httpx.post(VM_URL + "/v1/admin/keys", headers=ADMIN_HEADERS, json={"user": user, "plan": plan}, timeout=REQUEST_TIMEOUT)
    issued = response.json()
    issued_keys[user] = issued
    labor_key_rows.append({"상태 코드": response.status_code, "key_id": issued["key_id"], "사용자": issued["user"], "요금제": issued["plan"]})
display(pd.DataFrame(labor_key_rows))

# 노무 부하에서 사용할 클라이언트. 각 클라이언트에 해당 사용자의 서비스 키가 담깁니다.
user_clients = {}
for user in labor_users:
    user_clients[user] = httpx.AsyncClient(base_url=VM_URL, headers={"X-API-Key": issued_keys[user]["api_key"]}, timeout=REQUEST_TIMEOUT)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM이 뜬 직후의 첫 상담은 평소보다 오래 걸릴 수 있습니다(GPU에 따라 몇 초에서 수십 초). 이 느린 한 건이 부하에 섞이지 않게 먼저 한 건을 보냅니다.</p>
</div>

In [ ]:
# 데움 요청: pro 키 하나로 노무 질문 한 건을 보냅니다
warm_response = httpx.post(VM_URL + "/v1/labor/chat", headers={"X-API-Key": issued_keys["labor-app"]["api_key"]}, json={"question": labor_questions[0]["question"]}, timeout=REQUEST_TIMEOUT)

# 이 한 건이 서빙 API 의 누적 히스토그램에 어떻게 남았는지 봅니다
warm_snapshot = read_metrics(SERVING_METRICS_URL, ADMIN_HEADERS)
warm_view = {
    "상태 코드": warm_response.status_code,
    "걸린 시간(초)": round(warm_response.elapsed.total_seconds(), 1),
    "누적 요청 수(_count)": metric_sum(warm_snapshot, "serving_request_seconds_count", path="/v1/labor/chat"),
    "누적 걸린 시간 합(_sum)": round(metric_sum(warm_snapshot, "serving_request_seconds_sum", path="/v1/labor/chat"), 1),
}
display(warm_view)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">데움 요청도 서버의 누적값에 남습니다. 이번 부하만 보려면 요청을 보내기 전과 모두 받은 뒤의 값 차이를 계산해야 합니다.</p>
</div>

### 4-2. 부하 보내기와 두 스냅샷

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">부하는 FastAPI 서빙 API 단원 교안 01 5절의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">run_limited</code>를 그대로 쓰고, 실패도 한 행으로 남기는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">send_chat</code>을 더합니다.</p>
</div>

In [ ]:
import asyncio
import time

# 동시에 보낼 요청 수(서빙 벤치마킹 단원의 기본 부하 16)
MAX_CONCURRENCY = 16
semaphore = asyncio.Semaphore(MAX_CONCURRENCY)


async def run_limited(task):
    """동시에 실행할 요청 수를 제한하고, 한 요청이 끝나면 그 결과를 돌려줍니다.

    task에는 실행할 비동기 요청 함수가 들어옵니다. semaphore 자리가 날 때까지 기다린 뒤
    task를 실행하고, 요청 함수가 돌려준 결과를 반환합니다. 요청 오류는 여기서 감추지 않고 그대로 전달합니다.
    """
    async with semaphore:
        return await task


async def send_chat(client, path, body):
    """서비스 키가 설정된 비동기 client로 질문을 보내고 요청 결과 한 행을 돌려줍니다.

    path는 /v1/labor/chat 같은 상담 주소이고, body는 question을 담은 딕셔너리입니다.
    반환 딕셔너리에는 HTTP 상태 코드와 내 PC에서 잰 걸린 시간이 들어갑니다.
    상태 코드가 200이면 답의 끝난 이유, 출력 토큰 수, 답 글자 수도 넣습니다.
    200이 아니면 서버가 돌려준 오류 코드를 넣어, 429·502·504 응답도 다음 질문을 계속 보낼 수 있습니다.
    네트워크 연결 자체가 끊기거나 응답을 JSON으로 읽지 못한 오류는 그대로 전달됩니다.
    """
    started = time.perf_counter()
    response = await client.post(path, json=body)
    seconds = time.perf_counter() - started

    reply = response.json()
    row = {"상태 코드": response.status_code, "걸린 시간(초)": round(seconds, 2)}
    if response.status_code == 200:
        row["끝난 이유"] = reply["finish_reason"]
        row["출력 토큰"] = reply["usage"]["completion_tokens"]
        # 생각이 생성 상한을 다 쓰면 답(answer)이 None 으로 온다. 그때 답 글자 수는 0
        row["답 글자 수"] = 0
        if reply["answer"] is not None:
            row["답 글자 수"] = len(reply["answer"])
    else:
        row["오류"] = reply["error"]["code"]
    return row

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 노무 질문 64건을 최대 16개씩 동시에 보내 서빙 API 응답과 vLLM 상태를 측정합니다. 요청별 상태 코드·응답 시간과 부하 전후 메트릭으로 구간 요청 수·15초 이하 비율·출력 토큰·운영 원가를 계산하고, 부하 중 vLLM 스냅샷으로 대기 요청과 KV 캐시 사용률을 봅니다.</p>
</div>

In [ ]:
from datetime import datetime, timezone

# 부하에 쓸 노무 질문 64건. 사용할 키 셋은 4-1에서 만들었습니다.
load_questions = labor_questions[:64]

# 부하 전 두 스냅샷과 시작 시각(UTC)
before_api = read_metrics(SERVING_METRICS_URL, ADMIN_HEADERS)
before_vllm = read_metrics(VLLM_METRICS_URL)
load_start = datetime.now(timezone.utc)

# 질문 i 번을 키 i % 3 번에 맡겨 동시 MAX_CONCURRENCY 로 보냅니다
tasks = []
for i, item in enumerate(load_questions):
    user = labor_users[i % 3]
    tasks.append(run_limited(send_chat(user_clients[user], "/v1/labor/chat", {"question": item["question"]})))
# 요청 코루틴을 함께 예약합니다. run_limited 가 실제 동시 요청 수를 제한합니다.
load_job = asyncio.gather(*tasks)

# 10초 동안 현재 코드가 쉬는 사이 이벤트 루프가 예약한 상담 요청을 처리합니다.
await asyncio.sleep(10)
# 동기 메트릭 조회를 별도 스레드에서 실행합니다. 결과를 기다리는 동안에도 상담 요청은 진행됩니다.
during_vllm = await asyncio.to_thread(read_metrics, VLLM_METRICS_URL)

# 요청 64건이 모두 끝나기를 기다린 뒤 부하 뒤 두 스냅샷과 끝 시각, 구간 초를 기록합니다.
load_rows = await load_job
after_api = read_metrics(SERVING_METRICS_URL, ADMIN_HEADERS)
after_vllm = read_metrics(VLLM_METRICS_URL)
load_end = datetime.now(timezone.utc)
window_seconds = (load_end - load_start).total_seconds()

load_table = pd.DataFrame(load_rows)
print("부하 구간(초):", round(window_seconds, 1))
display(load_table)

### 4-3. 카운터 차이

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">부하 뒤 값에서 부하 전 값을 빼면 이번 부하 구간에 늘어난 양입니다. 예를 들어 노무 창구의 200 줄은 부하 전에 1(4-1의 데움 요청)이고, 64건이 모두 200이면 부하 뒤에 65가 되어 구간 값은 65 - 1 = 64입니다. 서빙 API에서는 상태 코드별로 이 차이를 계산합니다.</p>
</div>

In [ ]:
# 부하 뒤 스냅샷의 노무 창구 상태 코드마다, 부하 전 값을 뺀 구간 요청 수
status_rows = []
for row in after_api.itertuples():
    if row.name == "serving_requests_total" and row.labels["path"] == "/v1/labor/chat":
        status = row.labels["status"]
        count = row.value - metric_sum(before_api, "serving_requests_total", path="/v1/labor/chat", status=status)
        status_rows.append({"상태 코드": status, "구간 요청 수": count})
display(pd.DataFrame(status_rows))

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서빙 API가 401이나 429로 거절한 요청은 vLLM에 전달되지 않으므로 서빙 API 메트릭에만 남습니다. 사용자가 받은 실패는 이 층에서 확인합니다.</p>
<p style="margin:6px 0;">vLLM은 상태 코드 대신 요청이 끝난 이유(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">finished_reason</code>)마다 셉니다.</p>
</div>

In [ ]:
# vLLM 이 끝낸 요청을 끝난 이유마다 셉니다(부하 뒤 - 부하 전)
reason_counts = {}
for reason in ["stop", "length", "abort", "error", "repetition"]:
    after = metric_sum(after_vllm, "vllm:request_success_total", finished_reason=reason)
    before = metric_sum(before_vllm, "vllm:request_success_total", finished_reason=reason)
    reason_counts[reason] = after - before
display(reason_counts)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm:request_success_total</code>은 이름과 달리 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">abort</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">error</code>도 포함합니다. 사용자가 받은 실패는 서빙 API의 상태 코드로 구분합니다.</p>
<p style="margin:6px 0;">출력 토큰은 세 곳에서 셀 수 있습니다. 서빙 API의 창구별 카운터, vLLM의 생성 토큰 카운터, 내 PC가 받은 응답의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code>입니다.</p>
</div>

In [ ]:
# 세 곳에서 센 이번 구간의 출력 토큰
api_output_tokens = metric_sum(after_api, "serving_output_tokens_total", desk="labor") - metric_sum(before_api, "serving_output_tokens_total", desk="labor")
vllm_output_tokens = metric_sum(after_vllm, "vllm:generation_tokens_total") - metric_sum(before_vllm, "vllm:generation_tokens_total")
client_output_tokens = load_table["출력 토큰"].sum()

token_check = {
    "서빙 API(serving_output_tokens_total)": api_output_tokens,
    "vLLM(vllm:generation_tokens_total)": vllm_output_tokens,
    "내 PC(응답의 usage 합)": client_output_tokens,
}
display(token_check)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM은 같은 GPU Pod가 처리한 요청 전체를 세고, 서빙 API는 창구별로 나눠 셉니다. vLLM의 토큰 수가 더 크면 같은 시간에 다른 요청이 있었는지 확인합니다.</p>
</div>

### 4-4. 누적 비율과 구간 비율

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">컨테이너를 켠 뒤 200 응답이 많이 쌓였다면, 이번 부하에서 실패가 나도 전체 누적 200 응답 비율은 거의 바뀌지 않을 수 있습니다. 이번 부하의 200 응답 비율은 부하 전후 요청 수의 차이로 계산합니다.</p>
</div>

In [ ]:
# 누적: 컨테이너가 뜬 뒤 노무 창구가 센 모든 요청(3-1 의 401, 데움 요청 포함)
cumulative_total = metric_sum(after_api, "serving_requests_total", path="/v1/labor/chat")
cumulative_200 = metric_sum(after_api, "serving_requests_total", path="/v1/labor/chat", status="200")

# 구간: 부하 전후의 차이
window_total = cumulative_total - metric_sum(before_api, "serving_requests_total", path="/v1/labor/chat")
window_200 = cumulative_200 - metric_sum(before_api, "serving_requests_total", path="/v1/labor/chat", status="200")

ratio_table = pd.DataFrame([
    {"범위": "누적(컨테이너 시작부터)", "요청 수": cumulative_total, "200 비율": round(cumulative_200 / cumulative_total, 3)},
    {"범위": "구간(이번 부하)", "요청 수": window_total, "200 비율": round(window_200 / window_total, 3)},
])
display(ratio_table)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">누적값에는 이번 부하 이전에 보낸 요청도 포함됩니다. 과거 200 응답이 많이 쌓이면 이번 부하에서 실패가 나도 전체 누적 200 응답 비율은 거의 내려가지 않을 수 있습니다. 이번 부하만 보려면 부하 전후 카운터 차이로 계산합니다.</p>
</div>

## 5. 포화 신호와 캐시 적중

### 5-1. 포화를 보여 주는 세 게이지

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">포화(saturation)</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">GPU가 동시에 처리하는 자리나 KV 캐시 메모리 같은 자원이 가득 차서, 새 요청이 처리되지 못하고 기다리기 시작한 정도입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 지연이 길어진 원인이 서버가 일을 많이 받아서인지(포화), 다른 곳인지를 가릅니다. 포화면 동시 요청을 줄이거나 GPU를 늘리고, 아니면 다른 원인을 찾습니다.</li>
</ul>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/15_saturation_signals.png" width="1050" alt="포화 신호 그림: vLLM이 처리 중인 요청, 줄 선 대기 요청, KV 캐시 사용률이 차오르는 모습">

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM의 세 게이지가 포화를 보여 줍니다. 게이지는 읽은 순간의 값이라, 부하 도중에 찍은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">during_vllm</code>과 부하가 끝난 뒤의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">after_vllm</code>을 나란히 봅니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm:num_requests_running</code>:</strong> 지금 GPU에서 함께 생성 중인 요청 수입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm:num_requests_waiting</code>:</strong> 받았지만 아직 시작하지 못하고 줄 선 요청 수입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">vllm:kv_cache_usage_perc</code>:</strong> KV 캐시 공간 가운데 쓰는 비율입니다. 이름에 perc가 있지만 1이 100%인 0~1 값입니다.</li>
</ul>
</div>

In [ ]:
# 부하 도중과 부하 뒤의 세 게이지(값 그대로 읽습니다)
saturation_rows = []
for name in ["vllm:num_requests_running", "vllm:num_requests_waiting", "vllm:kv_cache_usage_perc"]:
    saturation_rows.append({
        "게이지": name,
        "부하 도중": round(metric_sum(during_vllm, name), 3),
        "부하 뒤": round(metric_sum(after_vllm, name), 3),
    })
saturation_table = pd.DataFrame(saturation_rows)
display(saturation_table)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">대기 요청:</strong> 측정한 순간의 값입니다. 0이어도 지연과 KV 캐시 사용률을 함께 살펴야 합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">KV 캐시 사용률:</strong> 1에 가까우면 새 요청이 쓸 캐시 공간이 부족할 수 있습니다. GPU 메모리 전체 점유율과 구분합니다.</li>
</ul>
</div>

### 5-2. 접두어 캐시 적중 비율

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">접두어 캐시 적중 비율은 조회 대상으로 확인한 입력 토큰 중 이전 요청에서 계산해 둔 값을 다시 사용한 토큰의 비율입니다. 적중이 낮으면 새로 계산할 입력 토큰이 늘어, 같은 부하에서도 첫 토큰 지연이 길어지고 포화에 더 빨리 닿을 수 있습니다.</p>
<p style="margin:6px 0;">접두어 캐시 적중은 서빙 벤치마킹 단원 교안 02와 같은 뜻입니다. vLLM은 입력 토큰 가운데 캐시에서 찾아본 수(queries)와 찾은 수(hits)를 셉니다.</p>
<p style="margin:6px 0;">예를 들어 노무 질문 두 건은 같은 시스템 메시지로 시작하고 질문만 다릅니다. 두 번째 요청은 시스템 메시지 부분의 토큰을 캐시에서 찾아(hits) 그대로 쓰고, 질문 부분만 새로 계산합니다.</p>
</div>

<div class="day79-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">접두어 캐시 적중 비율</strong> = 구간의 hits 차이 ÷ 구간의 queries 차이</div>

In [ ]:
# prefix_cache_queries_total: 접두어 캐시에서 조회 대상으로 확인한 토큰 수
prefix_queries = metric_sum(after_vllm, "vllm:prefix_cache_queries_total") - metric_sum(before_vllm, "vllm:prefix_cache_queries_total")
# prefix_cache_hits_total: 조회 대상 중 접두어 캐시에서 재사용한 토큰 수
prefix_hits = metric_sum(after_vllm, "vllm:prefix_cache_hits_total") - metric_sum(before_vllm, "vllm:prefix_cache_hits_total")
# 적중 토큰 수를 조회 대상 토큰 수로 나눠 비율을 구합니다
prefix_hit_ratio = prefix_hits / prefix_queries
print("조회한 토큰:", prefix_queries, "| 재사용한 토큰:", prefix_hits, "| 적중 비율:", round(prefix_hit_ratio, 3))

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">적중 비율이 낮아지면 같은 부하에서도 입력을 새로 계산하는 일이 늘어납니다. 교안 03은 이 변화로 장애 원인을 찾습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prefix_queries</code>가 0이면 저장한 템플릿의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--enable-prefix-caching</code> 설정을 확인합니다.</p>
</div>

## 6. 지연 분포

### 6-1. 응답 시간 경계별 요청 수

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">15초 경계의 값은 이번 부하에서 15초 이하로 끝난 요청 수입니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bucket_counts</code>는 경계마다 &quot;경계 이하 누적 건수&quot;가 이번 구간에 늘어난 양을 돌려줍니다. 칸 하나(앞 경계 초과, 이 경계 이하)의 건수는 이 값과 앞 행 값의 차이입니다.</p>
</div>

In [ ]:
def bucket_counts(before, after, name, **labels):
    """부하 전후 스냅샷으로 히스토그램 각 경계 이하의 요청 수를 계산해 표로 돌려줍니다.

    before와 after는 각각 부하를 보내기 전과 모두 받은 뒤의 /metrics 표입니다.
    name에는 _bucket을 뺀 메트릭 이름을 넣고, path처럼 창구 라벨을 추가할 수 있습니다.
    히스토그램의 경계별 숫자는 경계 이하 요청을 누적해 세므로, 각 경계에서 부하 전 숫자를
    부하 뒤 숫자에서 빼면 이번 부하의 경계 이하 건수가 됩니다. 표에는 경계와 누적 건수가 들어갑니다.
    """
    # 이 히스토그램의 경계(le 라벨)를 /metrics 에 적힌 순서대로 모읍니다
    edges = []
    for row in after.itertuples():
        if row.name == name + "_bucket" and row.labels["le"] not in edges:
            edges.append(row.labels["le"])

    rows = []
    for edge in edges:
        count = metric_sum(after, name + "_bucket", le=edge, **labels) - metric_sum(before, name + "_bucket", le=edge, **labels)
        rows.append({"경계(초)": edge, "경계 이하 누적 건수": count})
    return pd.DataFrame(rows)

In [ ]:
# 서빙 API 노무 창구의 요청 시간: 경계별 누적 건수의 구간 차이
api_buckets = bucket_counts(before_api, after_api, "serving_request_seconds", path="/v1/labor/chat")
display(api_buckets)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">+Inf</code>는 모든 요청을 포함합니다. 다만 5초 초과 10초 이하인 요청이 정확히 6초였는지 9초였는지는 히스토그램으로 알 수 없습니다.</p>
</div>

### 6-2. 경계 이하 비율

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">히스토그램은 경계마다 그 경계 이하인 요청 수를 그대로 세어 두므로, 경계 이하인 요청의 비율은 어림이 아니라 정확한 값입니다. 3-3의 네 건 예라면 15초 이하 비율은 3 / 4 = 0.75입니다. 2-1에서 15초를 칸 경계에 넣어 둔 이유입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/14_histogram_window.png" width="1050" alt="히스토그램 비율 그림: 15초 경계 막대 높이를 전체 막대로 나눈 비율은 정확하고, p95는 칸 안 어딘가까지만 안다">

<div class="day79-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">15초 이하 비율</strong> = 경계 15.0의 누적 건수 ÷ 경계 +Inf의 누적 건수</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">vLLM의 첫 토큰 지연(TTFT)은 1초 경계로 봅니다. 서빙 벤치마킹 단원 교안 01 5-3에서 정한 지연 목표(TTFT 1,000ms)입니다.</p>
</div>

In [ ]:
# 경계값("5.0", "15.0", "+Inf")을 행 이름으로 바꿔 누적 건수를 꺼냅니다
api_edges = api_buckets.set_index("경계(초)")["경계 이하 누적 건수"]
# 이번 부하의 서빙 API 요청 중 15초 이하인 수 / 전체 요청 수
within_15_ratio = api_edges["15.0"] / api_edges["+Inf"]

# vLLM 의 첫 토큰 지연(TTFT) 히스토그램에서 경계별 누적 건수를 읽습니다
ttft_buckets = bucket_counts(before_vllm, after_vllm, "vllm:time_to_first_token_seconds")
ttft_edges = ttft_buckets.set_index("경계(초)")["경계 이하 누적 건수"]
# 이번 부하의 TTFT 측정 중 1초 이하인 수 / 전체 TTFT 측정 수
ttft_within_1_ratio = ttft_edges["1.0"] / ttft_edges["+Inf"]

# 서빙 API의 15초·5초 기준과 vLLM의 1초 기준을 건수·전체·비율로 비교합니다
edge_table = pd.DataFrame([
    {"층": "서빙 API", "경계": "요청 시간 15초", "경계 이하": api_edges["15.0"], "전체": api_edges["+Inf"], "경계 이하 비율": round(within_15_ratio, 3)},
    {"층": "서빙 API", "경계": "요청 시간 5초", "경계 이하": api_edges["5.0"], "전체": api_edges["+Inf"], "경계 이하 비율": round(api_edges["5.0"] / api_edges["+Inf"], 3)},
    {"층": "vLLM", "경계": "TTFT 1초", "경계 이하": ttft_edges["1.0"], "전체": ttft_edges["+Inf"], "경계 이하 비율": round(ttft_within_1_ratio, 3)},
])
display(edge_table)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">15초 이하 비율이 1이면 모든 요청이 15초 안에 끝났습니다. 칸 경계에 없는 12초 이하 비율은 정확히 셀 수 없으므로 목표값을 서버의 경계에 넣어 두어야 합니다.</p>
<p style="margin:6px 0;">히스토그램은 요청마다 걸린 시간을 보존하지 않으므로 p95 같은 분위수는 칸의 범위로만 알 수 있습니다. 반면 칸 경계인 15초 이하 비율은 정확히 셀 수 있어 교안 03에서 응답 시간 목표를 판정할 때 씁니다.</p>
</div>

### 6-3. 두 층의 차이

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 요청의 걸린 시간을 세 곳이 각자 측정합니다. 내 PC는 요청을 보내고 답을 받기까지, 서빙 API는 요청을 받아 응답을 돌려주기까지, vLLM은 요청을 받아 마지막 토큰을 만들기까지입니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/13_two_layer_metrics.png" width="1050" alt="세 곳 측정 그림: 내 PC, 서빙 API, vLLM이 같은 요청에서 측정하는 시간의 범위와 그 차이가 생기는 곳">

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">평균은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_sum</code>의 차이를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_count</code>의 차이로 나눕니다.</p>
</div>

In [ ]:
# 평균 = 구간의 걸린 시간 합 / 구간의 요청 수
api_mean_seconds = (metric_sum(after_api, "serving_request_seconds_sum", path="/v1/labor/chat") - metric_sum(before_api, "serving_request_seconds_sum", path="/v1/labor/chat")) / (metric_sum(after_api, "serving_request_seconds_count", path="/v1/labor/chat") - metric_sum(before_api, "serving_request_seconds_count", path="/v1/labor/chat"))
vllm_e2e_mean_seconds = (metric_sum(after_vllm, "vllm:e2e_request_latency_seconds_sum") - metric_sum(before_vllm, "vllm:e2e_request_latency_seconds_sum")) / (metric_sum(after_vllm, "vllm:e2e_request_latency_seconds_count") - metric_sum(before_vllm, "vllm:e2e_request_latency_seconds_count"))

# 세 곳에서 측정한 이번 구간의 평균 걸린 시간(내 PC 는 200 응답만)
client_mean_seconds = load_table[load_table["상태 코드"] == 200]["걸린 시간(초)"].mean()
layer_table = pd.DataFrame([
    {"측정한 곳": "내 PC(send_chat)", "평균(초)": round(client_mean_seconds, 2)},
    {"측정한 곳": "서빙 API(serving_request_seconds)", "평균(초)": round(api_mean_seconds, 2)},
    {"측정한 곳": "vLLM(vllm:e2e_request_latency_seconds)", "평균(초)": round(vllm_e2e_mean_seconds, 2)},
])
display(layer_table)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서빙 API와 vLLM 평균의 차이:</strong> GPU 서버와의 통신과 VM 처리 시간이 포함됩니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">내 PC와 서빙 API 평균의 차이:</strong> PC와 VM 사이의 네트워크 시간이 포함됩니다.</li>
</ul>
</div>

## 7. 처리량과 운영 원가

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">서빙 벤치마킹 단원 교안 01 6절의 <strong style="font-size:1.13em;color:#1f3a5f;">100만 토큰당 비용</strong>은 GPU가 한 시간 내내 그 동시 요청 수로 쉬지 않고 일한다고 본 값입니다. 운영에서는 요청이 없는 시간에도 요금이 붙으므로, 실제로 흐른 시간의 요금을 실제로 만든 토큰으로 나눕니다. 이 교안은 이 값을 <strong style="font-size:1.13em;color:#1f3a5f;">100만 출력 토큰당 운영 원가</strong>라고 부르고, 두 이름을 섞지 않습니다.</p>
<p style="margin:6px 0;">예를 들어 시간당 0.57달러인 파드가 한 시간에 출력 토큰 100만 개를 만들면 운영 원가는 0.57달러입니다. 같은 한 시간에 요청이 적어 50만 개만 만들면 요금은 그대로라 원가는 1.14달러가 됩니다.</p>
</div>

<div class="day79-formula" style="margin:10px 0;padding:10px 16px;background:#f3f6fa;border-left:4px solid #1f3a5f;border-radius:4px;font-size:17px;line-height:1.8;color:#202124;"><strong style="color:#1f3a5f;">구간 출력 처리량</strong> = 구간 출력 토큰 ÷ 구간 초<br><strong style="color:#1f3a5f;">구간 GPU 요금</strong> = 시간당 요금 × 구간 초 ÷ 3600<br><strong style="color:#1f3a5f;">100만 출력 토큰당 운영 원가</strong> = 구간 GPU 요금 ÷ 구간 출력 토큰 × 1,000,000</div>

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">구간 출력 토큰은 vLLM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">vllm:generation_tokens_total</code> 차이(GPU가 실제로 만든 토큰)입니다. 교안 03도 같은 값으로 원가를 계산합니다.</p>
<p style="margin:6px 0;">아래에서 <strong style="font-size:1.13em;color:#1f3a5f;">창</strong>은 원가를 계산할 때 넣는 시간 범위입니다. 지금까지 쓴 부하 구간(부하 시작부터 끝까지)도 창 하나이고, 교안 03은 이 말로 목표를 판정하는 시간 범위를 정합니다. 부하 구간만 창으로 잡으면 운영 원가가 서빙 벤치마킹 단원의 100만 토큰당 비용과 같아집니다. 쉬는 시간이 들어가면 어떻게 바뀌는지 보려고, 같은 부하가 한 시간에 한 번만 왔다고 본 창을 함께 계산합니다.</p>
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_PRICE_PER_HOUR</code>는 1-1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 적은 시간당 요금입니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 부하 구간과 한 시간 운영 창의 출력 처리량과 GPU 원가를 비교합니다.</p>
</div>

<img style="display:block;max-width:100%;height:auto;margin:0;" src="../images/16_cost_window.png" width="1050" alt="원가 창 그림: 같은 출력 토큰이라도 부하 구간만 넣은 창과 쉬는 시간까지 넣은 한 시간 창은 GPU 요금이 달라 운영 원가가 달라짐">

In [ ]:
GPU_PRICE_PER_HOUR = float(os.environ["GPU_PRICE_PER_HOUR"])  # 시간당 요금(달러). 1-1에서 .env 에 적은 Total / hour 값(교안 02, 03도 같은 값을 읽습니다)

# 구간 출력 처리량 = vLLM 이 이번 구간에 만든 토큰(4-3 의 vllm_output_tokens) / 구간 초
output_tokens_per_second = vllm_output_tokens / window_seconds

# 100만 출력 토큰당 운영 원가 = 그 창의 GPU 요금 / 그 창의 출력 토큰 * 1_000_000
window_cost_per_million = GPU_PRICE_PER_HOUR * window_seconds / 3600 / vllm_output_tokens * 1_000_000
hour_cost_per_million = GPU_PRICE_PER_HOUR / vllm_output_tokens * 1_000_000   # 창이 한 시간이면 GPU 요금은 시간당 요금 그대로

cost_table = pd.DataFrame([
    {"창": "부하 구간만(" + str(round(window_seconds)) + "초)", "출력 토큰": vllm_output_tokens, "GPU 요금(달러)": round(GPU_PRICE_PER_HOUR * window_seconds / 3600, 4), "100만 출력 토큰당 운영 원가(달러)": round(window_cost_per_million, 3)},
    {"창": "한 시간에 이 부하 한 번", "출력 토큰": vllm_output_tokens, "GPU 요금(달러)": GPU_PRICE_PER_HOUR, "100만 출력 토큰당 운영 원가(달러)": round(hour_cost_per_million, 3)},
])
print("구간 출력 처리량(토큰/초):", round(output_tokens_per_second, 1))
display(cost_table)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 토큰을 만들어도 쉬는 시간이 측정 범위에 들어가면 원가가 커집니다. 교안 02에서 단가를 계산할 때 쓸 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output_tokens_per_second</code> 값은 적어 둡니다.</p>
</div>

## 8. 🖐️ 함께 따라하기: 전월세 창구의 구간 계산

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">본실습의 부하와 구간 계산을 전월세 창구에 적용해, 상태 코드별 요청 수, 15초 이하 비율, 100만 출력 토큰당 운영 원가를 직접 작성해 확인합니다.</p>
<p style="margin:6px 0;">전월세 키 발급과 부하 코드는 이 절에서만 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">본실습과 다른 점</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">주소와 라벨:</strong> 전월세 창구 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/lease/chat</code>입니다. 메트릭의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">path</code> 라벨도 이 주소입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">데이터:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/따라하기/lease_questions.jsonl</code>의 앞 32건입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">키:</strong> 이 절에서 발급할 pro 키 둘(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease-app</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">lease-web</code>)에 16건씩 맡깁니다.</li>
</ul>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">실습 진행 방법:</strong> 작성 셀의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">...</code>만 주석에 맞춰 채웁니다. 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">설정:</strong> 동시 요청 수는 본실습과 같은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">MAX_CONCURRENCY</code>(16), 시간당 요금은 7절의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_PRICE_PER_HOUR</code>입니다.</p>
</div>

### 8-1. 전월세 부하 보내기

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">먼저 전월세 창구용 키 두 개를 발급합니다. 키 원문은 변수에만 두고 표에는 키 번호와 사용자만 표시합니다.</p>
</div>

In [ ]:
# 함께 따라하기에 쓸 전월세 사용자 둘의 pro 키를 발급합니다.
lease_users = ["lease-app", "lease-web"]
lease_keys = {}
lease_key_rows = []
for user in lease_users:
    response = httpx.post(VM_URL + "/v1/admin/keys", headers=ADMIN_HEADERS, json={"user": user, "plan": "pro"}, timeout=REQUEST_TIMEOUT)
    issued = response.json()
    lease_keys[user] = issued["api_key"]
    lease_key_rows.append({"상태 코드": response.status_code, "key_id": issued["key_id"], "사용자": issued["user"], "요금제": issued["plan"]})
display(pd.DataFrame(lease_key_rows))

# 전월세 사용자별 서비스 키를 보관하는 클라이언트를 만듭니다.
for user in lease_users:
    user_clients[user] = httpx.AsyncClient(base_url=VM_URL, headers={"X-API-Key": lease_keys[user]}, timeout=REQUEST_TIMEOUT)

In [ ]:
# 전월세 질문 32건과 pro 키 둘
lease_questions = load_jsonl(data_dir / "따라하기" / "lease_questions.jsonl")

# 부하 전 두 스냅샷과 시작 시각
lease_before_api = read_metrics(SERVING_METRICS_URL, ADMIN_HEADERS)
lease_before_vllm = read_metrics(VLLM_METRICS_URL)
lease_start = datetime.now(timezone.utc)

# run_limited 로 감싸야 MAX_CONCURRENCY(16)을 지킵니다. 4-2의 send_chat 호출 모양을 참고하세요.
# 함께 따라하기: 질문 i 번을 키 i % 2 번의 클라이언트로 전월세 창구에 보내세요.
lease_tasks = []
for i, item in enumerate(lease_questions[:32]):
    user = lease_users[i % 2]
    lease_tasks.append(...)
lease_rows = await asyncio.gather(*lease_tasks)

# 부하 뒤 두 스냅샷과 끝 시각, 구간 초
lease_after_api = read_metrics(SERVING_METRICS_URL, ADMIN_HEADERS)
lease_after_vllm = read_metrics(VLLM_METRICS_URL)
lease_end = datetime.now(timezone.utc)
lease_window_seconds = (lease_end - lease_start).total_seconds()
display(pd.DataFrame(lease_rows))

### 8-2. 상태 코드별 구간 요청 수

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 4-3의 첫 셀을 참고합니다.
세부구현: 라벨 path 의 값은 전월세 창구 주소 문자열입니다. 부하 뒤 행을 고르는 조건과 부하 전 값을 빼는 metric_sum 의 path 인자에 같은 값을 넣습니다.
```

</details>

In [ ]:
# 함께 따라하기: 전월세 창구 주소로 상태 코드별 구간 요청 수를 채우세요
lease_status_rows = []
for row in lease_after_api.itertuples():
    if row.name == "serving_requests_total" and row.labels["path"] == ...:
        status = row.labels["status"]
        count = row.value - metric_sum(lease_before_api, "serving_requests_total", path=..., status=status)
        lease_status_rows.append({"상태 코드": status, "구간 요청 수": count})
display(pd.DataFrame(lease_status_rows))

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">전월세 주소는 이번에 처음 부르므로, 부하 전 카운터 값은 0입니다.</p>
</div>

### 8-3. 15초 이하 비율과 운영 원가

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 6-2와 7절의 식을 참고합니다.
세부구현: 15초 이하 비율은 경계 15.0 의 누적 건수를 경계 +Inf 의 누적 건수로 나눈 값입니다. 운영 원가는 시간당 요금에 구간 초를 곱해 3600 으로 나눈 요금을 구간 출력 토큰으로 나누고 1_000_000 을 곱합니다.
```

</details>

In [ ]:
# 전월세 창구 요청 시간의 경계별 누적 건수(경계를 인덱스로)
lease_buckets = bucket_counts(lease_before_api, lease_after_api, "serving_request_seconds", path="/v1/lease/chat")
lease_edges = lease_buckets.set_index("경계(초)")["경계 이하 누적 건수"]

# 함께 따라하기: 15초 이하 비율을 채우세요
lease_within_15_ratio = ...

# 이번 구간에 vLLM 이 만든 출력 토큰
lease_output_tokens = metric_sum(lease_after_vllm, "vllm:generation_tokens_total") - metric_sum(lease_before_vllm, "vllm:generation_tokens_total")

# 함께 따라하기: 부하 구간의 100만 출력 토큰당 운영 원가를 채우세요
lease_cost_per_million = ...

lease_summary = {
    "구간 초": round(lease_window_seconds, 1),
    "15초 이하 비율": round(lease_within_15_ratio, 3),
    "구간 출력 토큰": lease_output_tokens,
    "100만 출력 토큰당 운영 원가(달러)": round(lease_cost_per_million, 3),
}
display(lease_summary)

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">같은 GPU와 부하를 써도 창구별 시스템 메시지와 답 길이에 따라 계산한 값이 달라질 수 있습니다.</p>
</div>

## 9. 핵심 정리

<div class="day79-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">시작 상태:</strong> 저장한 템플릿으로 GPU Pod 하나를 만듭니다. VM에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">compose.yaml</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>만 두고, 코드를 push하면 GitHub Actions가 GHCR 이미지를 배포합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">메트릭 붙이기:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prometheus_client</code>의 Counter와 Histogram을 미들웨어 한 곳에서 세고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/metrics</code>는 관리자 키로만 엽니다. 세는 주소(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">CHAT_PATHS</code>)를 정해 줄 수가 늘지 않게 하고, 목표로 쓸 경계(15초)를 칸 경계에 넣습니다. 바뀐 파일은 commit, push로 배포 파이프라인이 배포합니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">세 종류와 구간:</strong> 카운터(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_total</code>)와 히스토그램(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_bucket{le}</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_sum</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">_count</code>)은 누적값이라 부하 전후 두 스냅샷의 차이로 구간 값을 만들고, 게이지는 읽은 순간의 값을 씁니다. 이전에 200 응답이 많이 쌓이면 최근 부하의 실패가 전체 누적 200 응답 비율에 작게 반영될 수 있습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">포화와 지연:</strong> 대기 요청이 0보다 크거나 KV 캐시 사용률이 1에 가까우면 포화입니다. 경계 이하 비율(15초 이하, TTFT 1초 이하)은 칸 경계에서 정확하고, 분위수는 칸 범위까지만 압니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">두 층:</strong> 서빙 API는 사용자가 받은 상태 코드와 창구별 토큰을, vLLM은 포화, 캐시 적중, 끝난 이유를 셉니다. 429처럼 vLLM까지 가지 않는 실패는 서빙 API에만 남고, 두 층의 평균 시간 차이는 중계와 VM 처리 몫입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">운영 원가:</strong> 100만 출력 토큰당 운영 원가 = 창의 GPU 요금 / 창의 출력 토큰 x 1,000,000입니다. 쉬는 시간이 창에 들어갈수록 서빙 벤치마킹 단원의 100만 토큰당 비용(쉬지 않고 일한다고 본 값)보다 커집니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 03으로 가져갈 숫자:</strong> 15초 이하 비율(응답 지연), TTFT 1초 이하 비율(첫 토큰 지연), 상태 코드별 구간 요청 수(가용성), 100만 출력 토큰당 운영 원가입니다. 교안 03은 여기에 답이 비지 않은 비율(품질)을 더해 목표를 정합니다. 대기 요청, KV 캐시 사용률, 접두어 캐시 적중 비율은 이 숫자가 나빠졌을 때 원인을 좁히는 데 씁니다.</li>
</ul>
<p style="margin:6px 0;">GPU 파드는 <strong style="font-size:1.13em;color:#1f3a5f;">켜 둔 채</strong> 교안 02로 갑니다. 교안 02는 같은 서빙 API에 요청 한 건씩의 기록(Langfuse trace)을 붙입니다.</p>
</div>